# 중간고사 대체 실습과제 — 문제 3

## 구조적 사고: MECE와 로직 트리

| 항목 | 내용 |
|------|------|
| **관련 챕터** | Ch03. 구조적 사고: MECE와 로직 트리 |
| **핵심 개념** | MECE 원칙, Why Tree, How Tree |
| **배점** | 25점 |
| **제출** | 이 노트북(.ipynb)을 실행 결과 포함하여 제출 |

---

## 시나리오

**프레시밀** CEO가 긴급 회의를 소집했습니다.

> *"매출이 하락하고 있습니다. 어디서 문제가 생겼는지 구조적으로 분석하고,
> 고객 이탈 원인을 정리해주세요."*

매출 구조 데이터와 이탈 사유 데이터가 제공됩니다.

In [1]:
# 환경설정
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# 데이터 로드
revenue = pd.read_csv('data/revenue_breakdown.csv')
churn = pd.read_csv('data/churn_reasons.csv')

print('=== 매출 구조 ===')
print(revenue.to_string(index=False))
print(f'\n총 매출: {revenue["Value_억원"].sum()}억 원')
print(f'\n=== 이탈 사유 (3개월 합계) ===')
print(churn.groupby('Reason_Category')['Churn_Count'].sum().sort_values(ascending=False).to_string())

=== 매출 구조 ===
Level1 Level2 Level3  Value_억원  Growth_pct
   온라인    구독형   정기배송        45         -12
   온라인    구독형   맞춤식단        28           5
   온라인   단건주문    자사앱        35          -8
   온라인   단건주문  배달앱입점        22         -25
  오프라인    자판기    사무실         8          15
  오프라인    자판기    헬스장         5          20
  오프라인  팝업스토어    백화점        12          -5
  오프라인  팝업스토어    대학교         6          10

총 매출: 161억 원

=== 이탈 사유 (3개월 합계) ===
Reason_Category
서비스품질    77
제품       30
가격       19
경쟁사      12


---

## 과제 1. MECE 검증과 매출 분해 (9점)

### 요구사항

1. **(3점)** 매출 데이터의 분류(Level1 → Level2 → Level3)가 **MECE를 만족하는지** 판단하세요.
   - 상호 배타적(ME)인가? 전체 포괄적(CE)인가?
   - 빠진 항목이 있다면 어떤 것인지 서술

2. **(3점)** 매출 데이터를 **Sunburst 차트** 또는 **Treemap**으로 시각화하세요.
   - Level1 → Level2 → Level3 계층 구조가 보여야 합니다

3. **(3점)** 성장률이 마이너스인 항목만 추출하여, 이 항목들의 매출이 전체에서 차지하는 **비율(%)**을 계산하고 `print()`로 출력하세요.

In [7]:
# ✏️ 과제 1-1: Sunburst 또는 Treemap 시각화
import plotly.express as px
import pandas as pd

# 데이터 로드 및 전처리
data = {
    'Level1': ['온라인', '온라인', '온라인', '온라인', '오프라인', '오프라인', '오프라인', '오프라인'],
    'Level2': ['구독형', '구독형', '단건주문', '단건주문', '자판기', '자판기', '팝업스토어', '팝업스토어'],
    'Level3': ['정기배송', '맞춤식단', '자사앱', '배달앱입점', '사무실', '헬스장', '백화점', '대학교'],
    'Value_억원': [45, 28, 35, 22, 8, 5, 12, 6],
    'Growth_pct': [-12, 5, -8, -25, 15, 20, -5, 10]
}
df = pd.DataFrame(data)

# Treemap 생성
fig = px.treemap(
    df, 
    path=['Level1', 'Level2', 'Level3'], 
    values='Value_억원',
    color='Growth_pct',
    color_continuous_scale='Greys', # 검정/회색 기조 유지
    title='프레시밀 매출 구조 및 성장률 현황 (단위: 억 원 / %)',
    hover_data={'Growth_pct': ':.1f'}
)

# 텍스트 및 마커 스타일링 (요청 규칙 적용)
fig.update_traces(
    textfont_color='black',            # 모든 텍스트 검정색
    marker=dict(
        line=dict(color='black', width=1.5), # 검정 테두리
        pad=dict(b=5, l=5, r=5, t=25)        # 가독성 여백
    ),
    textinfo="label+value+percent parent"    # 항목명, 매출액, 부모 대비 비중 표시
)

# 레이아웃 설정 (흰색 배경)
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()

In [8]:
# ✏️ 과제 1-2: 마이너스 성장 항목 추출 + 비율 계산
import pandas as pd

# 1. 데이터 로드
data = {
    'Level1': ['온라인', '온라인', '온라인', '온라인', '오프라인', '오프라인', '오프라인', '오프라인'],
    'Level2': ['구독형', '구독형', '단건주문', '단건주문', '자판기', '자판기', '팝업스토어', '팝업스토어'],
    'Level3': ['정기배송', '맞춤식단', '자사앱', '배달앱입점', '사무실', '헬스장', '백화점', '대학교'],
    'Value_억원': [45, 28, 35, 22, 8, 5, 12, 6],
    'Growth_pct': [-12, 5, -8, -25, 15, 20, -5, 10]
}
df = pd.DataFrame(data)

# 2. 성장률이 마이너스(< 0)인 항목만 추출
negative_growth_items = df[df['Growth_pct'] < 0]

# 3. 전체 매출 및 마이너스 항목 매출 합계 계산
total_revenue = df['Value_억원'].sum()
negative_revenue_sum = negative_growth_items['Value_억원'].sum()

# 4. 비율(%) 계산
negative_ratio = (negative_revenue_sum / total_revenue) * 100

# 결과 출력
print(f"=== 분석 결과 ===")
print(f"1. 전체 매출액: {total_revenue}억 원")
print(f"2. 마이너스 성장 항목 매출 합계: {negative_revenue_sum}억 원")
print(f"3. 전체 매출 대비 마이너스 성장 항목 비중: {negative_ratio:.2f}%")

=== 분석 결과 ===
1. 전체 매출액: 161억 원
2. 마이너스 성장 항목 매출 합계: 114억 원
3. 전체 매출 대비 마이너스 성장 항목 비중: 70.81%


### ✏️ 과제 1-3: MECE 검증 답안 (이 셀에 직접 작성)

- ME(상호 배타) 충족 여부: (충족한다.온라인과 오프라인, 구독형과 단건주문, 각 세부 채널 간의 분류 경계가 명확하다.)
- CE(전체 포괄) 충족 여부: (미흡한다.현재 분류 체계는 프레시밀의 전체 잠재적 매출 채널을 모두 포괄하지 못한다.)
- 보완이 필요한 점: (1.판매처를 저 확장시켜 외부쇼핑 매출을 추가시킨다. 2.기타 수입 반영: 배송비, 멤버십 회비 등 서비스 수익을 포함시켜 실제 매출액과 데이터가 일맥상통해야한다.)

---

## 과제 2. Why Tree: 이탈 원인 분석 (8점)

### 요구사항

1. **(3점)** 이탈 사유를 **Reason_Category별로 합계와 비율(%)**을 계산하여 출력하세요.

2. **(5점)** 아래 형식의 **Why Tree**를 작성하고, **수평 막대 차트**로 시각화하세요.

```
왜 고객이 이탈하는가? (총 138건)
├─ 서비스품질 (77건, 55.8%)
│   ├─ 배송 지연 (42건)
│   └─ 포장 파손 (35건)
├─ 제품 (30건, 21.7%)
│   ├─ 신선도 저하 (20건)
│   └─ 메뉴 다양성 부족 (10건)
├─ 가격 (19건, 13.8%)
│   └─ 구독료 상승 부담 (19건)
└─ 경쟁사 (12건, 8.7%)
    └─ 타사 프로모션 이동 (12건)
...
```

In [9]:
# ✏️ 과제 2-1: 카테고리별 합계 + 비율 계산
import pandas as pd

# 1. 데이터 로드 (이전 단계에서 제공된 데이터 기준)
# churn = pd.read_csv('data/churn_reasons.csv') 

# 2. 카테고리별 합계 계산
churn_summary = churn.groupby('Reason_Category')['Churn_Count'].sum().reset_index()

# 3. 내림차순 정렬
churn_summary = churn_summary.sort_values(by='Churn_Count', ascending=False)

# 4. 비율(%) 계산
total_churn = churn_summary['Churn_Count'].sum()
churn_summary['Ratio_pct'] = (churn_summary['Churn_Count'] / total_churn * 100).round(1)

# 5. 결과 출력
print("=== 이탈 사유 카테고리별 분석 ===")
print(churn_summary.to_string(index=False))

=== 이탈 사유 카테고리별 분석 ===
Reason_Category  Churn_Count  Ratio_pct
          서비스품질           77       55.8
             제품           30       21.7
             가격           19       13.8
            경쟁사           12        8.7


In [12]:
#  ✏️ 과제 2-2: Why Tree 텍스트 출력 + 수평 막대 차트 시각화
import plotly.express as px
import pandas as pd

# 데이터 구성 (과제 2-1 합계 결과 반영)
churn_data = {
    'Reason_Category': ['서비스품질', '제품', '가격', '경쟁사'],
    'Churn_Count': [77, 30, 19, 12],
    'Ratio_pct': [55.8, 21.7, 13.8, 8.7]
}
df_churn = pd.DataFrame(churn_data)
df_churn = df_churn.sort_values(by='Churn_Count', ascending=True) # 위에서 아래로 큰 순서대로 나오게 정렬

# 시각화 생성
fig = px.bar(
    df_churn,
    x='Churn_Count',
    y='Reason_Category',
    orientation='h',
    text='Churn_Count',
    title='<b>프레시밀 고객 이탈 원인 분석 (Why Tree 기반)</b>'
)

# 시각화 공통 규칙 및 스타일링 적용
fig.update_traces(
    textposition='outside',
    textfont=dict(color='black'),
    marker=dict(
        color='white',                # 흰색 배경
        line=dict(color='black', width=2) # 검정 테두리
    )
)

fig.update_layout(
    plot_bgcolor='white',             # 배경 흰색
    paper_bgcolor='white',
    font=dict(color='black'),         # 모든 텍스트 검정
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',        # 회색 기조 선
        title='이탈 건수 (건)'
    ),
    yaxis=dict(title='이탈 사유 카테고리'),
    margin=dict(l=100, r=50, t=80, b=50)
)

fig.show()

### ✏️ 과제 2 해석 (이 셀에 직접 타이핑)

- 이탈 원인 1위 카테고리와 그 비율은?
1위는 서비스품질이고 그 비율은 77%다.
- 이 카테고리 안에서 가장 건수가 많은 세부 사유는 무엇이며, 이를 줄이기 위해 가장 먼저 해야 할 일은? (1~2문장)
가장 건수가 많은 사유는 배송지연이다. 배송 지연을 줄이기 위해선 배송 센터를 더 세부화시켜 지연을 줄이고 소비자에게 지연 관련 보상 정책을 시행해 소비자와의 신뢰감을 쌓아야한다.

---

## 과제 3. How Tree: 해결책 도출 (8점)

### 요구사항

1. **(5점)** 과제 2에서 가장 큰 이탈 원인 **1개**에 대해 **How Tree**를 마크다운으로 작성하세요.
   - 최소 2단계 깊이 (시간축: 단기/장기)

```
어떻게 [원인]을 해결할 것인가?
├─ 단기 대응 (1개월 내)
│   ├─ 행동 1
│   └─ 행동 2
└─ 장기 개선 (3개월)
    ├─ 행동 3
    └─ 행동 4
```

2. **(3점)** 가장 효과가 클 해결책 **1개**를 선정하고, 그 근거를 데이터(이탈 건수 등)를 인용하여 2문장으로 서술하세요.

### ✏️ 과제 3-1: How Tree (이 셀에 직접 작성)

```
어떻게 [서비스 품질(배송 지연)]을 해결할 것인가?
├─ 단기 대응 (1개월 내: 신뢰 회복 및 프로세스 안정화)
│   ├─ 배송 지연 보상 강화: 배송 지연 발생 시 쿠폰 혹은 보상 제도 강화
│   ├─ 피크타임 인력 충원: 출고량이 집중되는 시간대에 단기 물류 보조 인력 추가 배치
└─ 장기 개선 (3개월 이상: 인프라 고도화 및 품질 내재화)
    ├─ 물류 센터 운송관리시스템 강화: 데이터 기반 배송 노선 최적화로 배송 효율 증대
    ├─ 지역별 거점 세부화: 수도권 외 핵심 지역을 세부화 시켜 지연을 최소한 줄임
```

### ✏️ 과제 3-2: 최우선 해결책 + 근거

(가장 효과가 큰 해결책은 '배송 시스템 최적화 및 지연 보상 강화'이다.
 전체 이탈의 1위 원인이 '서비스 품질'이다. 그중에서도 가장 비중이 큰 '배송 지연'을 해결하는 것이 고객 이탈 방지에 가장 즉각적이고 중요한 수단이므로 지역별 거점을 세부화 시켜 배송시간을 줄이고 혹여나 지연을 시키면 그에 맞는 큰 보상을 제공한다.)

---

## 채점 기준

| 과제 | 배점 | 채점 포인트 |
|------|------|------------|
| 과제 1. MECE + 매출 분해 | 9점 | MECE 판단(3) + 시각화(3) + 마이너스 분석(3) |
| 과제 2. Why Tree | 8점 | 집계(3) + 트리+시각화(5) |
| 과제 3. How Tree | 8점 | 해결책 구조(5) + 최우선 행동(3) |
| **합계** | **25점** | |

---

## 참고: 예상 정답값

**과제 1:**
- 전체 매출: 161억 원
- 마이너스 성장 4개 항목 합계: 114억 원 → 전체의 **약 71%**
- 정기배송(45억, -12%)이 가장 큰 하락 항목

**과제 2 이탈 카테고리(3개월 합계):**

| 카테고리 | 건수 | 비율 |
|---------|------|------|
| 서비스품질 | 77건 | 56% |
| 제품 | 30건 | 22% |
| 가격 | 19건 | 14% |
| 경쟁사 | 12건 | 9% |

- 서비스품질이 56%로 1위 → 그 중 배송 지연(47건)이 최다